# 03. 센서 이미지 특징 추출기

**모듈**: M2 - 딥러닝 기초
**날짜**: 2026-03-09

M1에서 만든 카메라 센서 이미지에서 CNN으로 **특징 벡터**를 추출합니다.

```
카메라 이미지 (3, 64, 64)  →  CNN Encoder  →  특징 벡터 (16,)
```

**목표**: 비슷한 상황의 이미지 → 비슷한 벡터 (클러스터 형성)
- 장애물 가까운 이미지들 → 한 그룹
- 장애물 먼 이미지들 → 다른 그룹

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.manifold import TSNE

# macOS 한글 폰트
mpl.rcParams['font.family'] = 'AppleGothic'
mpl.rcParams['axes.unicode_minus'] = False

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
print(f'장치: {device}')

## 1. 학습용 센서 이미지 생성

M1의 CameraSensor를 활용해서 다양한 상황의 이미지를 대량 생성합니다.

3가지 상황 (클래스):
- **가까운 장애물** (거리 0.1~0.3)
- **중간 장애물** (거리 0.4~0.6)
- **먼 장애물 / 없음** (거리 0.7~1.0)

In [ ]:
def generate_camera_image(obstacle_distance):
    """M1 CameraSensor와 동일한 로직"""
    img = np.zeros((64, 64, 3), dtype=np.uint8)
    
    # 하늘
    img[:32, :, 2] = 180
    img[:32, :, 1] = 200
    # 바닥
    img[32:, :, 1] = 150
    img[32:, :, 0] = 50
    
    # 장애물
    if obstacle_distance < 0.8:
        size = int(10 + (1 - obstacle_distance) * 20)
        cx, cy = 32 + np.random.randint(-8, 8), 32  # 약간의 위치 변동
        y1 = max(0, cy - size)
        y2 = min(64, cy + size)
        x1 = max(0, cx - size // 2)
        x2 = min(64, cx + size // 2)
        img[y1:y2, x1:x2, 0] = 255
        img[y1:y2, x1:x2, 1] = 0
        img[y1:y2, x1:x2, 2] = 0
    
    # 노이즈
    noise = np.random.randint(-15, 15, img.shape, dtype=np.int16)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    
    return img

# 데이터 생성
n_per_class = 200
images = []
labels = []  # 0=가까움, 1=중간, 2=멀리

for label, (d_min, d_max) in enumerate([(0.1, 0.3), (0.4, 0.6), (0.7, 1.0)]):
    for _ in range(n_per_class):
        dist = np.random.uniform(d_min, d_max)
        img = generate_camera_image(dist)
        images.append(img)
        labels.append(label)

# NumPy → PyTorch 텐서
images_np = np.array(images)
labels_np = np.array(labels)

# (N, H, W, C) → (N, C, H, W), 정규화
images_tensor = torch.from_numpy(images_np).permute(0, 3, 1, 2).float() / 255.0
labels_tensor = torch.from_numpy(labels_np).long()

print(f'생성된 데이터: {images_tensor.shape}')
print(f'클래스별 {n_per_class}장 × 3 = {len(images)}장')

# 샘플 시각화
fig, axes = plt.subplots(3, 6, figsize=(14, 7))
class_names = ['가까움 (0.1~0.3)', '중간 (0.4~0.6)', '멀리 (0.7~1.0)']

for row, label in enumerate(range(3)):
    idx = np.where(labels_np == label)[0][:6]
    for col, i in enumerate(idx):
        axes[row, col].imshow(images[i])
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(class_names[label], fontsize=10, rotation=0, labelpad=100)

plt.suptitle('센서 이미지: 장애물 거리별 3클래스', fontweight='bold')
plt.tight_layout()
plt.show()

## 2. CNN Encoder 정의

이미지 → 16차원 특징 벡터를 추출하는 Encoder.
이것이 M5에서 만들 **VLM Encoder의 원형**입니다.

```
(3, 64, 64) → Conv1(16) → Pool → Conv2(32) → Pool → FC → (16,)
```

In [ ]:
class CameraEncoder(nn.Module):
    """카메라 이미지 → 16차원 특징 벡터"""
    
    def __init__(self, feature_dim=16):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),    # (3,64,64) → (16,64,64)
            nn.ReLU(),
            nn.MaxPool2d(2),                    # → (16,32,32)
            nn.Conv2d(16, 32, 3, padding=1),   # → (32,32,32)
            nn.ReLU(),
            nn.MaxPool2d(2),                    # → (32,16,16)
            nn.Conv2d(32, 64, 3, padding=1),   # → (64,16,16)
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),            # → (64,4,4)
        )
        self.fc = nn.Sequential(
            nn.Flatten(),                       # → (1024,)
            nn.Linear(64 * 4 * 4, 64),
            nn.ReLU(),
            nn.Linear(64, feature_dim),         # → (16,)
        )
    
    def forward(self, x):
        features = self.features(x)
        return self.fc(features)

class ImageClassifier(nn.Module):
    """Encoder + 분류 헤드 (학습용)"""
    
    def __init__(self, feature_dim=16, num_classes=3):
        super().__init__()
        self.encoder = CameraEncoder(feature_dim)
        self.head = nn.Linear(feature_dim, num_classes)
    
    def forward(self, x):
        features = self.encoder(x)
        return self.head(features)
    
    def get_features(self, x):
        """특징 벡터만 반환"""
        return self.encoder(x)

model = ImageClassifier().to(device)
print(model)
print(f'\nEncoder 파라미터: {sum(p.numel() for p in model.encoder.parameters()):,}개')

# 테스트
test_input = images_tensor[:1].to(device)
features = model.get_features(test_input)
print(f'\n입력: {test_input.shape} → 특징: {features.shape}')

## 3. 분류 학습으로 Encoder 훈련

Encoder를 직접 학습하는 대신, **분류 과제**를 통해 간접적으로 학습합니다.
좋은 특징 벡터를 뽑아야 분류가 잘 되니까요.

In [ ]:
# 훈련/테스트 분할
n_train = int(len(images_tensor) * 0.8)
perm = torch.randperm(len(images_tensor))
train_idx, test_idx = perm[:n_train], perm[n_train:]

train_imgs = images_tensor[train_idx].to(device)
train_labels = labels_tensor[train_idx].to(device)
test_imgs = images_tensor[test_idx].to(device)
test_labels = labels_tensor[test_idx].to(device)

print(f'훈련: {len(train_imgs)}장, 테스트: {len(test_imgs)}장')

# 학습
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

losses = []
accuracies = []

for epoch in range(30):
    # 미니배치 학습
    model.train()
    perm = torch.randperm(len(train_imgs))
    epoch_loss = 0
    
    for i in range(0, len(train_imgs), 32):
        batch_idx = perm[i:i+32]
        batch_imgs = train_imgs[batch_idx]
        batch_labels = train_labels[batch_idx]
        
        outputs = model(batch_imgs)
        loss = criterion(outputs, batch_labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    
    # 평가
    model.eval()
    with torch.no_grad():
        test_out = model(test_imgs)
        acc = (test_out.argmax(1) == test_labels).float().mean().item()
    
    losses.append(epoch_loss / (len(train_imgs) // 32))
    accuracies.append(acc)
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:2d}: loss={losses[-1]:.4f}, 정확도={acc:.1%}')

print(f'\n최종 정확도: {accuracies[-1]:.1%}')

## 4. 특징 벡터 시각화 (t-SNE)

16차원 특징 벡터를 2D로 압축해서 시각화합니다.
**비슷한 이미지 → 비슷한 벡터 → 가까운 점**이면 성공!

In [ ]:
# 전체 데이터의 특징 벡터 추출
model.eval()
all_imgs = images_tensor.to(device)

with torch.no_grad():
    all_features = model.get_features(all_imgs).cpu().numpy()

print(f'특징 벡터: {all_features.shape}')

# t-SNE로 16D → 2D
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
features_2d = tsne.fit_transform(all_features)

# 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

class_names = ['가까움', '중간', '멀리']
colors = ['red', 'orange', 'green']

for label in range(3):
    mask = labels_np == label
    ax1.scatter(features_2d[mask, 0], features_2d[mask, 1],
               c=colors[label], label=class_names[label], alpha=0.6, s=30)

ax1.set_title('t-SNE: 특징 벡터 2D 투영')
ax1.legend()
ax1.set_xlabel('t-SNE 1')
ax1.set_ylabel('t-SNE 2')

# 학습 곡선
ax2.plot(losses, 'b-', label='Loss', linewidth=2)
ax2_twin = ax2.twinx()
ax2_twin.plot([a*100 for a in accuracies], 'g-', label='정확도', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss', color='blue')
ax2_twin.set_ylabel('정확도 (%)', color='green')
ax2.set_title('학습 곡선')

plt.suptitle('CNN Encoder: 이미지 → 특징 벡터 클러스터링', fontweight='bold')
plt.tight_layout()
plt.show()

print('빨강(가까움), 주황(중간), 초록(멀리)이 분리되면 Encoder가 잘 동작하는 것!')

## 5. 특징 벡터 유사도 확인

같은 클래스의 이미지 = 비슷한 벡터, 다른 클래스 = 다른 벡터인지 확인합니다.

In [ ]:
from torch.nn.functional import cosine_similarity

# 각 클래스에서 샘플 추출
model.eval()
samples = {}
for label in range(3):
    mask = labels_np == label
    idx = np.where(mask)[0][:5]
    with torch.no_grad():
        feats = model.get_features(images_tensor[idx].to(device))
    samples[label] = feats

# 클래스 간 코사인 유사도
print('=== 클래스 간 평균 코사인 유사도 ===')
print(f'{"":>10} {"가까움":>8} {"중간":>8} {"멀리":>8}')
for i, name_i in enumerate(class_names):
    sims = []
    for j in range(3):
        # 클래스 i의 모든 샘플과 클래스 j의 모든 샘플 비교
        sim_sum = 0
        count = 0
        for fi in samples[i]:
            for fj in samples[j]:
                sim_sum += cosine_similarity(fi.unsqueeze(0), fj.unsqueeze(0)).item()
                count += 1
        sims.append(sim_sum / count)
    print(f'{name_i:>10} {sims[0]:>8.3f} {sims[1]:>8.3f} {sims[2]:>8.3f}')

print()
print('대각선(같은 클래스) 값이 높고, 나머지가 낮으면 좋은 Encoder!')

## 6. M2 전체 정리: 파이프라인 연결

In [ ]:
print('=== M2 핵심 정리 ===')
print()
print('실습 1 (텐서): NumPy 센서 데이터 → PyTorch 텐서 변환')
print('  핵심: GPU 가속 + 자동 미분 = 학습 가능!')
print()
print('실습 2 (MNIST): CNN으로 이미지 분류 학습')
print('  핵심: Conv→Pool→FC 구조가 이미지 특징 추출의 기본!')
print()
print('실습 3 (특징 추출): 센서 이미지 → 16차원 특징 벡터')
print('  핵심: Encoder = 고차원 데이터 → 저차원 의미 벡터!')
print()
print('=== 파이프라인에서의 위치 ===')
print()
print('M1: SensorManager → 5개 센서 데이터 (NumPy)    [완료]')
print('M2: 텐서 변환 + CNN Encoder 기초              [완료]')
print('M3: Gymnasium 시뮬레이션 + LLM 연동            [다음]')
print('M4: LLM + V-JEPA + pi0 아키텍처 분석          ')
print('M5: 5개 센서별 Encoder (VLM+LLM+3종)          ')
print('M6: 5-Modal Fusion (5x16 → 64)                ')
print('M7: V-JEPA World Model (미래 예측)             ')
print('M8: pi0 Flow Matching (행동 생성)              ')
print('M9: 다양한 환경 실험 + LLM 명령               ')
print('M10: Triple Integration Capstone               ')